# VascuMap Batch Pipeline

Runs the full VascuMap pipeline on every image in a source folder.  
- **LIF files**: iterates through each image within the file  
- **TIF/TIFF files**: one pipeline run per file  

Set `record_timings = True` to save a per-image timing breakdown CSV alongside the outputs.

**Default outputs** (always saved):
| File | Description |
|------|-------------|
| `*_overlay_geometry_0.tif` | 2-D device overlay with inner/outer geometry |
| `*_skeleton_overview.png` | Skeleton overview plot |
| `*_analysis_metrics.csv` | Quantitative vascular network metrics |

**Extra outputs** (when `save_all_interim = True`):
| File | Description |
|------|-------------|
| `*_cropped_stack_aligned.npy` | Brightfield stack at 2 µm iso |
| `*_vessel_translation_aligned.npy` | Pix2Pix image translation |
| `*_clean_segmentation.npy` | Cleaned binary vessel segmentation |
| `*_skeleton.npy` | Skeleton from pruned graph |
| `*_holes.npy` | Binary pore mask |
| `*_hole_labels_per_slice.npy` | Per-slice labelled pores |
| `*_hole_distance_per_slice_um.npy` | Inscribed-radius distance map |
| `*_full_graph_skeleton.npy` | Pre-pruning skeleton |
| `*_vessel_mask.npy` | Raw vessel mask |
| `*_graph_nodes.npz` | Sprout/junction node coords |
| `*_clean_graph.pkl` | NetworkX graph (pickle) |

In [1]:
# ── Imports ────────────────────────────────────────────────────────────────────
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(r"C:\Users\taylorhearn\git_repos\vascumap\bel_vascumap")))

import numpy as np
import pandas as pd
import tifffile
from liffile import LifFile

from vascumap import VascuMap
from utils import resize_dask

W0330 13:31:05.751000 593768 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.0.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
c:\Users\taylorhearn\AppData\Local\miniconda3\envs\clean_vascumap\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ── Load models once (shared across all batch jobs) ────────────────────────────
from models import Pix2Pix, load_segmentation_model

shared_model_p2p = Pix2Pix(
    model_path=r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\epoch=117-val_g_psnr=20.47-val_g_ssim=0.62.ckpt"
)
shared_model_unet = load_segmentation_model(
    r"C:\Users\taylorhearn\git_repos\vascumap\luca_models\best_full.pth"
)
print("Models loaded.")

Models loaded.


In [3]:
# ── Job discovery helper ──────────────────────────────────────────────────────
def discover_jobs(source_dir: str, force_mask_central_region=None, require_merged: bool = True):
    """Discover .lif/.tif/.tiff files and return batch jobs.

    Parameters
    ----------
    force_mask_central_region : "light" | "dark" | False | None
        Override the per-filename mask mode for every job.
        None (default) → auto-detect from filename:
          - "marina" AND "bead" in name  → "light"  (bright organoid)
          - "marina" in name only        → "dark"   (dark organoid, invert first)
          - otherwise                    → False    (no masking)

    Returns
    -------
    source : Path
        Resolved source directory.
    image_files : list[Path]
        Matching image files found in source.
    jobs : list[tuple[Path, int, str|False]]
        (path, image_index, mask_mode) entries for batch execution.
    """
    source = Path(source_dir)
    if not source.is_dir():
        raise FileNotFoundError(f"source_dir does not exist: {source}")

    image_files = sorted(
        p for p in source.iterdir()
        if p.is_file() and p.suffix.lower() in (".lif", ".tif", ".tiff")
    )

    def _auto_mask_mode(p: Path, img_name: str = ""):
        # Combine file name and image name so per-image names (e.g. "Bead…") are checked
        name = (p.name + " " + img_name).lower()
        if "marina" in name and "bead" in name:
            return "light"
        if "marina" in name:
            return "dark"
        return False

    jobs = []
    for p in image_files:
        if p.suffix.lower() == ".lif":
            try:
                with LifFile(p) as lif:
                    n_images = len(lif.images)
                    for idx in range(n_images):
                        img_name = lif.images[idx].name if hasattr(lif.images[idx], "name") else ""
                        if require_merged and "merged" not in img_name.lower():
                            continue
                        should_mask = (
                            force_mask_central_region
                            if force_mask_central_region is not None
                            else _auto_mask_mode(p, img_name)
                        )
                        jobs.append((p, idx, should_mask))
            except Exception as exc:
                print(f"[SKIP] Could not inspect {p.name}: {exc}")
        else:
            should_mask = (
                force_mask_central_region
                if force_mask_central_region is not None
                else _auto_mask_mode(p)
            )
            if require_merged and "merged" not in p.name.lower():
                continue
            jobs.append((p, 0, should_mask))

    return source, image_files, jobs


In [4]:
# ── Processing + batch execution helpers ──────────────────────────────────────
def process_and_save(image_path: Path, image_index: int, output_base: str,
                     device_width_um: float,
                     hough_line_length: int = 400,
                     mask_central_region: bool = True,
                     save_all_interim: bool = False,
                     channel: int = 0,
                     model_p2p=None,
                     model_unet=None):
    """Run full VascuMap pipeline and save outputs to a per-image folder."""
    vm = VascuMap(
        use_device_segmentation_app=False,
        image_source_path=str(image_path),
        image_index=image_index,
        device_width_um=device_width_um,
        hough_line_length=hough_line_length,
        mask_central_region=mask_central_region,
        channel=channel,
        model_p2p=model_p2p,
        model_unet=model_unet,
    )

    if image_path.suffix.lower() == ".lif":
        vm.image_name = f"{image_path.stem}_img{image_index}"
    else:
        vm.image_name = image_path.stem

    output_dir = Path(output_base) / vm.image_name
    print(f"  Output → {output_dir}")
    vm.pipeline(output_dir=output_dir, save_all_interim=save_all_interim)
    return vm


def run_batch(jobs, output_base: str, device_width_um: float,
              hough_line_length: int = 400, channel: int = 0,
              save_all_interim: bool = False,
              model_p2p=None, model_unet=None,
              max_retries: int = 2, label: str = "Batch",
              record_timings: bool = False):
    """Run a list of jobs, print summary, and optionally save a timing CSV."""
    results = []
    timing_rows = []

    for i, (image_path, image_index, mask_flag) in enumerate(jobs, start=1):
        tag = f" (LIF #{image_index})" if image_path.suffix.lower() == ".lif" else ""
        print(f"\n{'='*70}")
        print(f"[{i}/{len(jobs)}] {image_path.name}{tag}  mask_central_region={mask_flag}")
        print(f"{'='*70}")

        last_exc = None
        for attempt in range(1, max_retries + 1):
            try:
                _t_job = time.time()
                vm = process_and_save(
                    image_path=image_path,
                    image_index=image_index,
                    output_base=output_base,
                    device_width_um=device_width_um,
                    hough_line_length=hough_line_length,
                    mask_central_region=mask_flag,
                    save_all_interim=save_all_interim,
                    channel=channel,
                    model_p2p=model_p2p,
                    model_unet=model_unet,
                )
                _t_wall = time.time() - _t_job
                results.append((vm.image_name or image_path.stem, "OK", ""))

                if record_timings:
                    timing_rows.append({
                        "image_name":     vm.image_name,
                        "source_file":    image_path.name,
                        "image_index":    image_index,
                        "status":         "OK",
                        "t_device_seg_s": round(getattr(vm, "_t_device_seg", 0), 1),
                        "t_preprocess_s": round(getattr(vm, "_t_preprocess", 0), 1),
                        "t_inference_s":  round(getattr(vm, "_t_inference", 0), 1),
                        "t_analysis_s":   round(getattr(vm, "_t_analysis", 0), 1),
                        "t_pipeline_s":   round(getattr(vm, "_t_total", 0), 1),
                        "t_job_wall_s":   round(_t_wall, 1),
                    })
                    # Save after every image so partial results survive crashes
                    pd.DataFrame(timing_rows).to_csv(
                        Path(output_base) / "batch_timings.csv", index=False
                    )

                last_exc = None
                break
            except Exception as exc:
                last_exc = exc
                if attempt < max_retries:
                    print(f"  ⚠ Attempt {attempt} failed: {exc}  — retrying...")
                else:
                    print(f"  ✗ FAILED after {max_retries} attempts: {exc}")

        if last_exc is not None:
            results.append((image_path.name + tag, "FAILED", str(last_exc)))
            if record_timings:
                timing_rows.append({
                    "image_name": image_path.name + tag,
                    "source_file": image_path.name,
                    "image_index": image_index,
                    "status": "FAILED",
                    "t_device_seg_s": None, "t_preprocess_s": None,
                    "t_inference_s": None, "t_analysis_s": None,
                    "t_pipeline_s": None, "t_job_wall_s": None,
                })
                pd.DataFrame(timing_rows).to_csv(
                    Path(output_base) / "batch_timings.csv", index=False
                )

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*70}")
    n_ok = sum(1 for _, s, _ in results if s == "OK")
    print(f"{label} complete: {n_ok}/{len(results)} succeeded")
    if any(s == "FAILED" for _, s, _ in results):
        print("\nFailed images:")
        for name, status, msg in results:
            if status == "FAILED":
                print(f"  - {name}: {msg}")

    if record_timings and timing_rows:
        csv_path = Path(output_base) / "batch_timings.csv"
        df = pd.DataFrame(timing_rows)
        df.to_csv(csv_path, index=False)
        print(f"\nTiming breakdown saved → {csv_path}")
        print(df[["image_name", "t_device_seg_s", "t_preprocess_s",
                   "t_inference_s", "t_analysis_s", "t_pipeline_s"]].to_string(index=False))

    return results

In [5]:
# ── Run configuration (edit here) ─────────────────────────────────────────────
source_dir    = r"Z:\Bel\Vascumap_Example_Lifs\original_images\test_subset"
output_base   = r"Z:\Bel\Vascumap_Speed_Test"

force_mask_central_region = None    # "dark" / "light" / False / None (auto by filename)
require_merged = False
save_all_interim = False
record_timings = True               # Save per-image timing breakdown CSV

device_width_um    = 35.0
hough_line_length  = 400
channel            = 0
max_retries        = 2

In [ ]:
# ── Discover jobs + run ────────────────────────────────────────────────────────
source, image_files, jobs = discover_jobs(
    source_dir=source_dir,
    force_mask_central_region=force_mask_central_region,
    require_merged=require_merged,
)

print(f"Source: {source}")
print(f"Files found: {len(image_files)}  |  Total jobs: {len(jobs)}")
if len(image_files) == 0:
    print("No .lif/.tif/.tiff files found in this folder.")
if len(jobs) == 0 and len(image_files) > 0:
    print("Files were found, but all were filtered out (likely by require_merged).")

for i, (p, idx, _) in enumerate(jobs, start=1):
    tag = f" (LIF image {idx})" if p.suffix.lower() == ".lif" else ""
    print(f"  {i}. {p.name}{tag}")

batch_results = run_batch(
    jobs=jobs,
    output_base=output_base,
    device_width_um=device_width_um,
    hough_line_length=hough_line_length,
    channel=channel,
    save_all_interim=save_all_interim,
    model_p2p=shared_model_p2p,
    model_unet=shared_model_unet,
    max_retries=max_retries,
    label="Batch",
    record_timings=record_timings,
)

Source: Z:\Bel\Vascumap_Example_Lifs\original_images\test_subset
Files found: 7  |  Total jobs: 7
  1. 20250619_Vascumap_Dev25_11_Daisy10.tif
  2. 20250619_Vascumap_Dev25_11_WicellWicellECFB10.tif
  3. Defne_placenta_270325_Dexa2_Device_2.tif
  4. Defne_placenta_270325_Gly_Device_6.tif
  5. Dorota_merged_stellaris_R1_007.tif
  6. Dorota_merged_stellaris_R1_017.tif
  7. farid_new_flow_double_channel.tif

[1/7] 20250619_Vascumap_Dev25_11_Daisy10.tif  mask_central_region=False
  [TIFF] Raw array shape: (50, 4693, 2859)  dtype=uint8
  [TIFF] Final stack shape: (50, 4693, 2859)
  [Dilation rescue] disk(3) found device but area is only 0.3% of image — still running Hough (device mask touches image border)
  [Hough fallback] Primary device detection failed (device mask touches image border) — using probabilistic Hough lines
  ⏱  Device segmentation: 47.3s
  Output → Z:\Bel\Vascumap_Speed_Test\20250619_Vascumap_Dev25_11_Daisy10
  Segmentation diagnostic plot → 20250619_Vascumap_Dev25_11_Daisy1

Processing chunks: 100%|██████████| 3/3 [00:26<00:00,  8.84s/it]


strong contiguous vote planes 14-18
  ⏱  Stage 2 (Pix2Pix + UNet): 785.5s
Running skeletonisation and analysis...


Processing chunks: 100%|██████████| 1/1 [00:01<00:00,  1.30s/it]


 chip_volume_um3  convex_hull_volume_um3  vessel_volume_um3  vessel_volume_fraction  total_vessel_length_um  vessel_length_per_chip_volume_um_inverse2  sprouts_per_vessel_length_um_inverse  junctions_per_vessel_length_um_inverse  skeleton_fractal_dimension  skeleton_lacunarity  median_sprout_and_branch_orientation_deg  p90_minus_p10_sprout_and_branch_orientation_deg  median_sprout_and_branch_median_cs_area_um2  p90_minus_p10_sprout_and_branch_median_cs_area_um2  median_junction_dist_nearest_junction_um  p90_minus_p10_junction_dist_nearest_junction_um  median_sprout_dist_nearest_endpoint_um  p90_minus_p10_sprout_dist_nearest_endpoint_um  average_vessel_volume_um3  total_internal_pore_count  internal_pore_area_fraction_in_filled_vascular_area  median_internal_pore_area_um2  p90_minus_p10_internal_pore_area_um2  median_internal_pore_max_inscribed_radius_um  p90_minus_p10_internal_pore_max_inscribed_radius_um  sprouts_per_chip_volume_um_inverse3  junctions_per_chip_volume_um_inverse3  tota

...

Processing chunks: 100%|██████████| 2/2 [00:16<00:00,  8.08s/it]


strong contiguous vote planes 0-7
  ⏱  Stage 2 (Pix2Pix + UNet): 475.8s
  Trimmed 32 top / 0 bottom over-segmented z-slices
Running skeletonisation and analysis...
  ⚠ Attempt 1 failed: range() arg 3 must not be zero  — retrying...
  [TIFF] Raw array shape: (12, 2335, 1442)  dtype=uint8
  [TIFF] Final stack shape: (12, 2335, 1442)
  [Dilation rescue] disk(3) found device but area is only 0.4% of image — still running Hough (device mask touches image border)
  [Hough fallback] Primary device detection failed (device mask touches image border) — using probabilistic Hough lines
  ⏱  Device segmentation: 5.8s
  Output → Z:\Bel\Vascumap_Speed_Test\Dorota_merged_stellaris_R1_017
  Segmentation diagnostic plot → Dorota_merged_stellaris_R1_017_segmentation_diagnostic.png
Initial z votes {0: 7, 1: 6, 2: 14, 3: 19, 4: 17, 5: 22, 6: 11, 7: 4, 8: 0, 9: 0, 10: 0, 11: 0}
First cropping to z: {0: 7, 1: 6, 2: 14, 3: 19, 4: 17, 5: 22, 6: 11, 7: 4, 8: 0, 9: 0, 10: 0, 11: 0}
Stack width 95.99885454545455

Processing chunks: 100%|██████████| 2/2 [00:16<00:00,  8.31s/it]


strong contiguous vote planes 0-7
  ⏱  Stage 2 (Pix2Pix + UNet): 482.3s
  Trimmed 32 top / 0 bottom over-segmented z-slices
Running skeletonisation and analysis...
  ✗ FAILED after 2 attempts: range() arg 3 must not be zero

[7/7] farid_new_flow_double_channel.tif  mask_central_region=False
  [TIFF] Raw array shape: (34, 2, 7435, 2856)  dtype=uint8
  [TIFF] 4-D → extracting channel 0 along axis 1 (shape=(34, 2, 7435, 2856))
  [TIFF] Final stack shape: (34, 7435, 2856)
  [Dilation rescue] disk(6) found device but area is only 1.8% of image — still running Hough (device mask touches image border)
  [Hough fallback] Primary device detection failed (device mask touches image border) — using probabilistic Hough lines
  ⏱  Device segmentation: 56.3s
  Output → Z:\Bel\Vascumap_Speed_Test\farid_new_flow_double_channel
  Segmentation diagnostic plot → farid_new_flow_double_channel_segmentation_diagnostic.png
Initial z votes {0: 0, 1: 0, 2: 0, 3: 1, 4: 1, 5: 2, 6: 0, 7: 0, 8: 2, 9: 2, 10: 2, 11:

 78%|███████▊  | 1201/1547 [03:29<00:58,  5.93it/s]